# Notebook 08 — Validation Set Inference & Report

Runs inference on the external validation set (94 handwritten/risky + 306 regular/safe) \
for **ResNet50**, **ViT-Base**, and **DiT** and produces a standalone interactive HTML \
report with a model-selector dropdown.

**Inputs:**
- `data/handwritten_validation_set.xlsx` — risky class (label = 1)
- `data/regular_documents_validation_set.xlsx` — safe class (label = 0)
- `data/validation_set/handwritten/` — PDF files for risky class
- `data/validation_set/regular_documents/` — PDF files for safe class
- `checkpoints/baseline/best_resnet50.pt` + `calibrator_resnet50.pkl`
- `checkpoints/baseline/best_vit_base_patch16_224.pt` + `calibrator_vit_base_patch16_224.pkl`
- `checkpoints/dit/best_model.pt` + `calibrator.pkl`

**Outputs:**
- `validation_report/report_data.json` — plot data nested under `models` key
- `validation_report/index.html` — standalone HTML report with model dropdown (zip-ready)

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.notebook import tqdm

from src.inference.predict import load_pipeline, predict_single
from src.inference.service_schema import RiskCategory

print(f'PyTorch: {torch.__version__}')
print(f'Root:    {ROOT}')

## 1 — Load Validation Manifests from XLSX

In [ ]:
HANDWRITTEN_XLSX = ROOT / 'data' / 'handwritten_validation_set.xlsx'
REGULAR_XLSX     = ROOT / 'data' / 'regular_documents_validation_set.xlsx'
HANDWRITTEN_DIR  = ROOT / 'data' / 'validation_set' / 'handwritten'
REGULAR_DIR      = ROOT / 'data' / 'validation_set' / 'regular_documents'

df_hw = pd.read_excel(HANDWRITTEN_XLSX)[['file name']].rename(columns={'file name': 'filename'})
df_hw['true_label'] = 1
df_hw['source']     = 'handwritten'
df_hw['pdf_path']   = df_hw['filename'].apply(lambda f: HANDWRITTEN_DIR / f)

df_reg = pd.read_excel(REGULAR_XLSX)[['file name']].rename(columns={'file name': 'filename'})
df_reg['true_label'] = 0
df_reg['source']     = 'regular_documents'
df_reg['pdf_path']   = df_reg['filename'].apply(lambda f: REGULAR_DIR / f)

df_all = pd.concat([df_hw, df_reg], ignore_index=True)
df_all['file_exists'] = df_all['pdf_path'].apply(lambda p: p.exists())
missing = df_all[~df_all['file_exists']]

print(f'Total rows: {len(df_all)}')
print(f'  Handwritten (risky, label=1): {len(df_hw)}')
print(f'  Regular     (safe,  label=0): {len(df_reg)}')
print(f'  Files found: {df_all["file_exists"].sum()} / {len(df_all)}')
if len(missing):
    print(f'  Missing ({len(missing)}):')
    for _, row in missing.iterrows():
        print(f'    {row["pdf_path"]}')

df_all.head(3)

## 2 — Model Configurations

In [ ]:
MODELS = {
    'ResNet50': {
        'checkpoint':  ROOT / 'checkpoints' / 'baseline' / 'best_resnet50.pt',
        'calibrator':  ROOT / 'checkpoints' / 'baseline' / 'calibrator_resnet50.pkl',
        'model_type': 'resnet50',
        'config':      ROOT / 'configs' / 'baseline.yaml',
    },
    'ViT-Base': {
        'checkpoint':  ROOT / 'checkpoints' / 'baseline' / 'best_vit_base_patch16_224.pt',
        'calibrator':  ROOT / 'checkpoints' / 'baseline' / 'calibrator_vit_base_patch16_224.pkl',
        'model_type': 'vit',
        'config':      ROOT / 'configs' / 'baseline.yaml',
    },
    'DiT': {
        'checkpoint':  ROOT / 'checkpoints' / 'dit' / 'best_model.pt',
        'calibrator':  ROOT / 'checkpoints' / 'dit' / 'calibrator.pkl',
        'model_type': 'dit',
        'config':      ROOT / 'configs' / 'dit.yaml',
    },
}

for name, cfg in MODELS.items():
    print(f'{name}: checkpoint exists={cfg["checkpoint"].exists()}, '
          f'calibrator exists={cfg["calibrator"].exists()}')

## 3 — Run Inference on All 400 PDFs (per model)

In [ ]:
all_results = {}   # model_name -> list[dict]

for model_name, cfg in MODELS.items():
    print(f'\n=== {model_name} ===')

    mdl, calibrator, thresholds, device = load_pipeline(
        checkpoint_path=str(cfg['checkpoint']),
        calibrator_path=str(cfg['calibrator']),
        model_type=cfg['model_type'],
        config_path=str(cfg['config']),
    )
    print(f'  device={device}, T_low={thresholds["T_low"]:.3f}, '
          f'T_high={thresholds["T_high"]:.3f}, T={calibrator.temperature:.4f}')

    results = []
    for _, row in tqdm(df_all.iterrows(), total=len(df_all), desc=f'  {model_name}'):
        if not row['file_exists']:
            results.append({
                'filename': row['filename'], 'source': row['source'],
                'true_label': row['true_label'],
                'probability': float('nan'), 'predicted_category': 'error',
                'raw_logit': float('nan'), 'error': 'file_not_found',
            })
            continue
        try:
            resp = predict_single(
                pdf_path=str(row['pdf_path']),
                model=mdl, calibrator=calibrator,
                thresholds=thresholds, dpi=150, device=device,
            )
            results.append({
                'filename': row['filename'], 'source': row['source'],
                'true_label': row['true_label'],
                'probability': resp.confidence,
                'predicted_category': resp.risk_category.value,
                'raw_logit': resp.raw_logit, 'error': None,
            })
        except Exception as exc:
            results.append({
                'filename': row['filename'], 'source': row['source'],
                'true_label': row['true_label'],
                'probability': float('nan'), 'predicted_category': 'error',
                'raw_logit': float('nan'), 'error': str(exc),
            })

    all_results[model_name] = results
    rdf = pd.DataFrame(results)
    print(f'  Completed: {len(rdf)}  Errors: {rdf["error"].notna().sum()}')
    # free GPU/MPS memory between models
    del mdl
    if torch.backends.mps.is_available(): torch.mps.empty_cache()

print('\nAll models done.')

## 4 — Compute Metrics per Model

In [ ]:
N_WORST = 10

model_metrics = {}   # model_name -> dict

for model_name, results in all_results.items():
    rdf = pd.DataFrame(results)
    valid_df = rdf[rdf['error'].isna()].copy()

    # Re-load thresholds from checkpoint (T_low for false-safe-rate)
    ck = torch.load(str(MODELS[model_name]['checkpoint']), map_location='cpu', weights_only=False)
    T_LOW  = ck['thresholds']['T_low']
    T_HIGH = ck['thresholds']['T_high']

    y_true = valid_df['true_label'].values
    y_prob = valid_df['probability'].values
    y_pred = (y_prob >= 0.5).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    cr = classification_report(y_true, y_pred,
                               target_names=['safe (0)', 'risky (1)'],
                               output_dict=True)

    risky_mask = y_true == 1
    fsr = float(((y_prob < T_LOW) & risky_mask).sum() / risky_mask.sum())

    # Worst cases
    safe_df  = valid_df[valid_df['true_label'] == 0].sort_values('probability', ascending=False)
    risky_df = valid_df[valid_df['true_label'] == 1].sort_values('probability', ascending=True)
    worst_fp = safe_df.head(N_WORST).copy()
    worst_fp['true_label_name'] = 'safe'
    worst_fn = risky_df.head(N_WORST).copy()
    worst_fn['true_label_name'] = 'risky'

    model_metrics[model_name] = {
        'valid_df': valid_df,
        'y_true': y_true, 'y_prob': y_prob, 'y_pred': y_pred,
        'cm': cm, 'cr': cr,
        'T_LOW': T_LOW, 'T_HIGH': T_HIGH, 'fsr': fsr,
        'worst_fp': worst_fp, 'worst_fn': worst_fn,
    }

    tn, fp, fn, tp = cm.ravel()
    acc = float((y_pred == y_true).mean())
    print(f'{model_name}: acc={acc:.1%}  FSR={fsr:.1%}  '
          f'TN={tn} FP={fp} FN={fn} TP={tp}')

## 5 — Build Plotly Figures per Model

In [ ]:
def build_confusion_matrix_fig(cm, model_name):
    tn, fp, fn, tp = cm.ravel()
    total = tn + fp + fn + tp
    z    = [[tn, fp], [fn, tp]]
    text = [
        [f'<b>{tn}</b><br>({tn/total:.1%})', f'<b>{fp}</b><br>({fp/total:.1%})'],
        [f'<b>{fn}</b><br>({fn/total:.1%})', f'<b>{tp}</b><br>({tp/total:.1%})'],
    ]
    fig = go.Figure(go.Heatmap(
        z=z,
        x=['Predicted: Safe', 'Predicted: Risky'],
        y=['True: Safe',      'True: Risky'],
        text=text, texttemplate='%{text}',
        textfont={'size': 18}, colorscale='Blues', showscale=False,
        hovertemplate='%{y} → %{x}<br>Count: %{z}<extra></extra>',
    ))
    fig.update_layout(
        title=dict(text=f'Confusion Matrix — {model_name}', font=dict(size=20), x=0.5),
        xaxis=dict(title='Predicted Label', side='bottom', tickfont=dict(size=13)),
        yaxis=dict(title='True Label', tickfont=dict(size=13)),
        height=460, plot_bgcolor='white', paper_bgcolor='white',
        margin=dict(l=80, r=40, t=80, b=60),
    )
    return fig


def build_classification_report_fig(cr, model_name):
    rows_to_show = ['safe (0)', 'risky (1)', 'macro avg', 'weighted avg']
    cr_rows = {k: cr[k] for k in rows_to_show if k in cr}
    header_vals = ['Class', 'Precision', 'Recall', 'F1-Score', 'Support']
    cell_vals = [
        list(cr_rows.keys()),
        [f"{v['precision']:.3f}" for v in cr_rows.values()],
        [f"{v['recall']:.3f}"    for v in cr_rows.values()],
        [f"{v['f1-score']:.3f}"  for v in cr_rows.values()],
        [int(v['support'])       for v in cr_rows.values()],
    ]
    n = len(cr_rows)
    fig = go.Figure(go.Table(
        header=dict(
            values=[f'<b>{h}</b>' for h in header_vals],
            fill_color='#1a1a2e', font=dict(color='white', size=14),
            align='center', height=36,
        ),
        cells=dict(
            values=cell_vals,
            fill_color=[['#f8f9fa' if i % 2 == 0 else 'white' for i in range(n)]] * 5,
            font=dict(color='#222', size=13), align='center', height=32,
        ),
    ))
    fig.update_layout(
        title=dict(text=f'Classification Report — {model_name}', font=dict(size=20), x=0.5),
        height=300, paper_bgcolor='white', margin=dict(l=20, r=20, t=80, b=20),
    )
    return fig


def build_histogram_fig(y_true, y_prob, T_LOW, T_HIGH, model_name):
    safe_probs  = y_prob[y_true == 0]
    risky_probs = y_prob[y_true == 1]
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=safe_probs, name='Safe (label=0)', nbinsx=30, opacity=0.75,
        marker_color='#2563eb',
        hovertemplate='Prob: %{x:.2f}<br>Count: %{y}<extra>Safe</extra>',
    ))
    fig.add_trace(go.Histogram(
        x=risky_probs, name='Risky (label=1)', nbinsx=30, opacity=0.75,
        marker_color='#dc2626',
        hovertemplate='Prob: %{x:.2f}<br>Count: %{y}<extra>Risky</extra>',
    ))
    fig.add_vline(x=T_LOW,  line_dash='dash', line_color='#f59e0b', line_width=2,
                  annotation_text=f'T_low={T_LOW:.3f}',
                  annotation_position='top right', annotation_font_size=11)
    fig.add_vline(x=T_HIGH, line_dash='dot',  line_color='#7c3aed', line_width=2,
                  annotation_text=f'T_high={T_HIGH:.3f}',
                  annotation_position='top right', annotation_font_size=11)
    fig.update_layout(
        title=dict(text=f'Probability Distribution — {model_name}', font=dict(size=20), x=0.5),
        xaxis=dict(title='Risk Probability', range=[0, 1], tickfont=dict(size=13)),
        yaxis=dict(title='Count', tickfont=dict(size=13)),
        barmode='overlay', legend=dict(font=dict(size=13), x=0.75, y=0.95),
        height=480, plot_bgcolor='white', paper_bgcolor='white',
        margin=dict(l=60, r=40, t=80, b=60),
    )
    fig.update_xaxes(showgrid=True, gridcolor='#e5e7eb')
    fig.update_yaxes(showgrid=True, gridcolor='#e5e7eb')
    return fig


def build_worst_cases_fig(worst_fp, worst_fn, model_name):
    def wc_table(df, color):
        n = len(df)
        return go.Table(
            header=dict(
                values=[f'<b>{h}</b>' for h in
                        ['Filename', 'Source', 'True Label', 'Risk Probability', 'Predicted Category']],
                fill_color='#1a1a2e', font=dict(color='white', size=13),
                align=['left','center','center','center','center'], height=34,
            ),
            cells=dict(
                values=[
                    df['filename'].tolist(), df['source'].tolist(),
                    df['true_label_name'].tolist(),
                    [f'{p:.3f}' for p in df['probability'].tolist()],
                    df['predicted_category'].tolist(),
                ],
                fill_color=[[
                    '#fef2f2' if i % 2 == 0 else 'white' for i in range(n)
                ] if color == 'red' else [
                    '#eff6ff' if i % 2 == 0 else 'white' for i in range(n)
                ]] * 5,
                font=dict(color='#222', size=12),
                align=['left','center','center','center','center'], height=30,
            ),
        )
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            f'False Positives — Safe docs with Highest Risk Score (top {N_WORST})',
            f'False Negatives — Risky docs with Lowest Risk Score (top {N_WORST})',
        ],
        vertical_spacing=0.12,
        specs=[[{'type': 'table'}], [{'type': 'table'}]],
    )
    fig.add_trace(wc_table(worst_fp, 'red'),  row=1, col=1)
    fig.add_trace(wc_table(worst_fn, 'blue'), row=2, col=1)
    fig.update_layout(
        title=dict(text=f'Worst Case Predictions — {model_name}', font=dict(size=20), x=0.5),
        height=800, paper_bgcolor='white', margin=dict(l=20, r=20, t=100, b=20),
    )
    return fig


model_figs = {}
for model_name, m in model_metrics.items():
    model_figs[model_name] = {
        'confusion_matrix':       build_confusion_matrix_fig(m['cm'], model_name),
        'classification_report':  build_classification_report_fig(m['cr'], model_name),
        'probability_histogram':  build_histogram_fig(
                                      m['y_true'], m['y_prob'],
                                      m['T_LOW'], m['T_HIGH'], model_name),
        'worst_cases':            build_worst_cases_fig(m['worst_fp'], m['worst_fn'], model_name),
    }
    print(f'{model_name}: figures built')

# Preview first model
model_figs['ResNet50']['confusion_matrix'].show()

## 6 — Export Report Data & Build HTML

In [ ]:
REPORT_DIR = ROOT / 'validation_report'
REPORT_DIR.mkdir(exist_ok=True)


def fig_to_dict(fig):
    return json.loads(pio.to_json(fig))


report_data = {'models': {}}

for model_name, m in model_metrics.items():
    tn, fp, fn, tp = m['cm'].ravel()
    acc = float((m['y_pred'] == m['y_true']).mean())
    summary = {
        'total':           int(len(m['valid_df'])),
        'n_safe':          int((m['valid_df']['true_label'] == 0).sum()),
        'n_risky':         int((m['valid_df']['true_label'] == 1).sum()),
        'accuracy':        acc,
        'false_safe_rate': m['fsr'],
        't_low':           float(m['T_LOW']),
        't_high':          float(m['T_HIGH']),
        'threshold_for_binary': 0.5,
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }
    figs = model_figs[model_name]
    report_data['models'][model_name] = {
        'summary':              summary,
        'confusion_matrix':     fig_to_dict(figs['confusion_matrix']),
        'classification_report':fig_to_dict(figs['classification_report']),
        'probability_histogram':fig_to_dict(figs['probability_histogram']),
        'worst_cases':          fig_to_dict(figs['worst_cases']),
    }

json_path = REPORT_DIR / 'report_data.json'
with open(json_path, 'w', encoding='utf-8') as fh:
    json.dump(report_data, fh, ensure_ascii=False, indent=2)

print(f'report_data.json written → {json_path}  ({json_path.stat().st_size / 1024:.0f} KB)')

In [ ]:
report_json_str = json.dumps(report_data, ensure_ascii=False)

HTML_TEMPLATE = '''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Validation Inference Report</title>
<script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
<style>
  :root {
    --bg: #f4f6fb; --surface: #ffffff; --primary: #1a1a2e;
    --accent: #2563eb; --accent2: #dc2626; --muted: #6b7280;
    --border: #e5e7eb;
  }
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    background: var(--bg); color: var(--primary); min-height: 100vh;
  }
  header {
    background: var(--primary); color: white;
    padding: 24px 40px 20px; border-bottom: 3px solid var(--accent);
  }
  header h1 { font-size: 1.6rem; font-weight: 700; letter-spacing: -0.5px; }
  header p  { font-size: 0.875rem; color: #94a3b8; margin-top: 4px; }
  .controls {
    background: var(--surface); border-bottom: 1px solid var(--border);
    padding: 14px 40px; display: flex; align-items: center; gap: 24px; flex-wrap: wrap;
  }
  .model-label { font-size: 0.85rem; color: var(--muted); font-weight: 600;
                 text-transform: uppercase; letter-spacing: 0.5px; }
  #modelSelect {
    padding: 8px 14px; border: 1px solid var(--border); border-radius: 6px;
    background: var(--bg); font-size: 0.9rem; color: var(--primary);
    cursor: pointer; outline: none; min-width: 180px;
  }
  #modelSelect:focus { border-color: var(--accent); }
  .summary-bar {
    display: flex; gap: 32px; flex-wrap: wrap; padding: 14px 40px;
    background: var(--surface); border-bottom: 1px solid var(--border);
  }
  .stat { display: flex; flex-direction: column; }
  .stat .val { font-size: 1.35rem; font-weight: 700; color: var(--accent); }
  .stat .lbl { font-size: 0.72rem; color: var(--muted);
               text-transform: uppercase; letter-spacing: 0.5px; }
  .tabs {
    display: flex; gap: 4px; padding: 16px 40px 0;
    background: var(--bg); border-bottom: 1px solid var(--border);
  }
  .tab-btn {
    padding: 10px 22px; border: 1px solid var(--border); border-bottom: none;
    background: #f3f4f6; color: var(--muted); font-size: 0.875rem;
    font-weight: 500; cursor: pointer; border-radius: 6px 6px 0 0; transition: all 0.15s;
  }
  .tab-btn:hover { background: #e0e7ff; color: var(--accent); }
  .tab-btn.active {
    background: var(--surface); color: var(--accent); font-weight: 600;
    border-bottom: 2px solid var(--surface); box-shadow: 0 -2px 0 var(--accent);
  }
  .pages { padding: 28px 40px 40px; }
  .page { display: none; }
  .page.active { display: block; }
  .plot-card {
    background: var(--surface); border-radius: 10px; border: 1px solid var(--border);
    box-shadow: 0 1px 4px rgba(0,0,0,0.06); padding: 28px 24px 20px;
  }
  .plot-card .caption {
    margin-top: 12px; font-size: 0.82rem; color: var(--muted); text-align: center;
  }
  .plotly-chart { width: 100%; }
  footer {
    text-align: center; padding: 20px; font-size: 0.78rem; color: var(--muted);
    border-top: 1px solid var(--border); margin-top: 20px;
  }
  @media (max-width: 640px) {
    header { padding: 14px 16px 12px; } header h1 { font-size: 1.15rem; }
    .controls { padding: 10px 16px; } .summary-bar { padding: 10px 16px; gap: 14px; }
    .stat .val { font-size: 1.1rem; } .stat .lbl { font-size: 0.68rem; }
    .tabs { padding: 10px 0 0; overflow-x: auto; flex-wrap: nowrap; }
    .tab-btn { white-space: nowrap; padding: 8px 14px; font-size: 0.8rem; flex-shrink: 0; }
    .pages { padding: 14px 12px 28px; }
    .plot-card { padding: 14px 10px 12px; border-radius: 8px; }
  }
</style>
</head>
<body>

<header>
  <h1>Validation Inference Report</h1>
  <p>Hebrew PDF Hallucination-Risk Classifier &mdash; External validation set (400 documents)</p>
</header>

<div class="controls">
  <span class="model-label">Model:</span>
  <select id="modelSelect" onchange="switchModel(this.value)">
    <option value="ResNet50">ResNet50</option>
    <option value="ViT-Base">ViT-Base</option>
    <option value="DiT">DiT (best stage)</option>
  </select>
</div>

<div class="summary-bar" id="summary-bar"></div>

<div class="tabs">
  <button class="tab-btn active" onclick="showPage(0, this)">Confusion Matrix</button>
  <button class="tab-btn" onclick="showPage(1, this)">Classification Report</button>
  <button class="tab-btn" onclick="showPage(2, this)">Probability Distribution</button>
  <button class="tab-btn" onclick="showPage(3, this)">Worst Cases</button>
</div>

<div class="pages">
  <div class="page active" id="page-0">
    <div class="plot-card">
      <div id="plot-cm" class="plotly-chart"></div>
      <p class="caption">Rows = true labels &bull; Columns = predicted labels &bull; Binary threshold at 0.5</p>
    </div>
  </div>
  <div class="page" id="page-1">
    <div class="plot-card">
      <div id="plot-cr" class="plotly-chart"></div>
      <p class="caption">Precision, Recall, F1-Score per class with macro and weighted averages</p>
    </div>
  </div>
  <div class="page" id="page-2">
    <div class="plot-card">
      <div id="plot-hist" class="plotly-chart"></div>
      <p class="caption">Dashed line = T_low (safe routing) &bull; Dotted line = T_high (risky routing)</p>
    </div>
  </div>
  <div class="page" id="page-3">
    <div class="plot-card">
      <div id="plot-wc" class="plotly-chart"></div>
      <p class="caption">Top 10 cases where the model is most wrong in each direction</p>
    </div>
  </div>
</div>

<footer>Generated by notebook 08_validation_inference.ipynb &bull; ResNet50 / ViT-Base / DiT checkpoints</footer>

<script>
const REPORT_DATA = __REPORT_DATA__;
const CONFIG = { responsive: true, displayModeBar: true };

let currentModel = 'ResNet50';

function statBar(s) {
  const items = [
    { val: s.total,                                   lbl: 'Total samples' },
    { val: s.n_safe,                                  lbl: 'Safe (label 0)' },
    { val: s.n_risky,                                 lbl: 'Risky (label 1)' },
    { val: (s.accuracy  * 100).toFixed(1) + '%',      lbl: 'Accuracy (thr 0.5)' },
    { val: (s.false_safe_rate * 100).toFixed(1) + '%',lbl: 'False-safe rate' },
    { val: s.t_low.toFixed(3),                        lbl: 'T_low' },
    { val: s.t_high.toFixed(3),                       lbl: 'T_high' },
  ];
  return items.map(i =>
    `<div class="stat"><span class="val">${i.val}</span><span class="lbl">${i.lbl}</span></div>`
  ).join('');
}

function renderModel(modelKey) {
  const m = REPORT_DATA.models[modelKey];
  document.getElementById('summary-bar').innerHTML = statBar(m.summary);
  Plotly.react('plot-cm',   m.confusion_matrix.data,         m.confusion_matrix.layout,         CONFIG);
  Plotly.react('plot-cr',   m.classification_report.data,    m.classification_report.layout,    CONFIG);
  Plotly.react('plot-hist', m.probability_histogram.data,    m.probability_histogram.layout,    CONFIG);
  Plotly.react('plot-wc',   m.worst_cases.data,              m.worst_cases.layout,              CONFIG);
}

function switchModel(modelKey) {
  currentModel = modelKey;
  renderModel(modelKey);
}

function showPage(idx, btn) {
  document.querySelectorAll('.page').forEach((p, i) => p.classList.toggle('active', i === idx));
  document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
  btn.classList.add('active');
}

// Initial render
renderModel(currentModel);
</script>
</body>
</html>'''

html_content = HTML_TEMPLATE.replace('__REPORT_DATA__', report_json_str)

html_path = REPORT_DIR / 'index.html'
with open(html_path, 'w', encoding='utf-8') as fh:
    fh.write(html_content)

print(f'index.html written \u2192 {html_path}  ({html_path.stat().st_size / 1024:.0f} KB)')
print()
print('validation_report/ directory contents:')
for p in sorted(REPORT_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1024:.0f} KB)')
print()
print('To share: zip -r validation_report.zip validation_report/')

## 7 — Summary

In [ ]:
print('=== External Validation Summary — All Models ===')
print(f'  Dataset: 400 files (306 safe, 94 risky)')
print()
header = f'  {"Model":<12} {"Acc":>7} {"F1":>7} {"FSR":>8}  {"CM"}'
print(header)
print('  ' + '-' * (len(header) - 2))

for model_name, m in model_metrics.items():
    tn, fp, fn, tp = m['cm'].ravel()
    acc = float((m['y_pred'] == m['y_true']).mean())
    cr  = m['cr']
    f1  = cr.get('macro avg', {}).get('f1-score', float('nan'))
    print(f'  {model_name:<12} {acc:>6.1%} {f1:>7.3f} {m["fsr"]:>7.1%}'
          f'  TN={tn} FP={fp} FN={fn} TP={tp}')

print()
print(f'Report: {(ROOT / "validation_report" / "index.html").resolve()}')
print('Use model dropdown to switch between ResNet50, ViT-Base, and DiT.')